In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 0 — Montar Drive
# Sube a tu Google Drive:
#   - modelo_placa.pt          (si ya entrenaste antes, para continuar)
#   - carpeta 'reales_placa/'  con fotos/videos del condominio (opcional)
#       ├── reales_placa/placas/   ← fotos de placas del condominio
#       └── reales_placa/videos/  ← videos de vehículos entrando
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path

RUTA_MODELO_BASE  = '/content/drive/MyDrive/modelo_placa.pt'
RUTA_REALES_DRIVE = '/content/drive/MyDrive/reales_placa'

if Path(RUTA_MODELO_BASE).exists():
    shutil.copy(RUTA_MODELO_BASE, '/content/modelo_placa_base.pt')
    print('✅ modelo_placa.pt copiado (continuará fine-tuning)')
else:
    print('ℹ️  Sin modelo base — se partirá desde YOLOv8n pre-entrenado en placas')

TIENE_REALES = Path(RUTA_REALES_DRIVE).exists()
print(f'📁 Datos reales en Drive: {TIENE_REALES}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 1 — Configurar Kaggle + instalar dependencias
#
# Antes de correr:
#   1. Clic en el ícono de llave (🔑) en Colab
#   2. Agregar secret: Name = KAGGLE_TOKEN, Value = tu token KGAT_...
#   3. Activar toggle 'Notebook access'
# ─────────────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = 'roly'
os.environ['KAGGLE_TOKEN']    = userdata.get('KAGGLE_TOKEN')

print('📦 Instalando dependencias...')
!pip install kaggle ultralytics easyocr -q
print('✅ Kaggle, Ultralytics y EasyOCR instalados')

import ultralytics, easyocr
print(f'   ultralytics: {ultralytics.__version__}')
print(f'   easyocr listo')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 2 — Descargar datasets de detección de placas
#
# Datasets (anotaciones YOLO o Pascal VOC — se convierten en celda 4):
#   - Number Plate Detection     ~3.000 imgs  (mixto internacional)
#   - Car Plate Detection        ~  433 imgs  (Pascal VOC, variado)
#   - Vehicle License Plate      ~1.200 imgs  (cámaras de seguridad)
#
# El modelo base keremberke ya detecta placas muy bien.
# El fine-tuning mejora precisión en cámaras de bajo ángulo / baja luz.
# ─────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path

print('📥 [1/3] Number Plate Detection (YOLO format)...')
!kaggle datasets download -d aslanahmedov/number-plate-detection \
    -p /content/ds_placas1 --unzip -q 2>/dev/null || \
 kaggle datasets download -d elysian19/number-plate-detection \
    -p /content/ds_placas1 --unzip -q 2>/dev/null || \
 echo '⚠️  Dataset 1 no disponible'
print('✅ Dataset 1 listo')

print('\n📥 [2/3] Car Plate Detection (Pascal VOC)...')
!kaggle datasets download -d andrewmvd/car-plate-detection \
    -p /content/ds_placas2 --unzip -q 2>/dev/null || \
 echo '⚠️  Dataset 2 no disponible'
print('✅ Dataset 2 listo')

print('\n📥 [3/3] Vehicle License Plate (cámara seguridad)...')
!kaggle datasets download -d snmahsa/vehicle-detection-and-license-plate-recognition \
    -p /content/ds_placas3 --unzip -q 2>/dev/null || \
 echo '⚠️  Dataset 3 no disponible — continuando sin él'

print('\n✅ Descarga completada')
for folder in ['/content/ds_placas1', '/content/ds_placas2', '/content/ds_placas3']:
    if Path(folder).exists():
        total = sum(1 for _ in Path(folder).rglob('*') if _.is_file())
        print(f'  {folder}: {total} archivos')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 3 — Extraer frames de videos reales del condominio (si los tienes)
#
# Sube a Drive/reales_placa/:
#   videos/  → videos .mp4/.avi de la cámara de entrada del condominio
#   placas/  → fotos directas de placas (sin necesidad de anotar)
#
# De cada video extrae 1 frame cada 15 (~2 fps a 30fps)
# Un video de 1 minuto → ~120 imágenes de contexto real
# ─────────────────────────────────────────────────────────────────────────────
import cv2, shutil
from pathlib import Path

EXTENSIONES_VIDEO  = {'.mp4', '.avi', '.mov', '.mkv'}
EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png'}
FRAME_CADA_N       = 15   # más denso que vehículos: las placas aparecen poco tiempo

def extraer_frames(ruta_video, carpeta_destino):
    cap = cv2.VideoCapture(str(ruta_video))
    extraidos, frame_idx = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_CADA_N == 0:
            out = carpeta_destino / f'real_{ruta_video.stem}_{frame_idx:06d}.jpg'
            cv2.imwrite(str(out), frame)
            extraidos += 1
        frame_idx += 1
    cap.release()
    return extraidos

total_reales = 0
dst_reales   = Path('/content/reales_placa')
dst_reales.mkdir(parents=True, exist_ok=True)

if TIENE_REALES:
    src_placas = Path(RUTA_REALES_DRIVE) / 'placas'
    if src_placas.exists():
        for img in src_placas.rglob('*'):
            if img.suffix.lower() in EXTENSIONES_IMAGEN:
                shutil.copy(img, dst_reales / img.name)
                total_reales += 1
        print(f'✅ {total_reales} fotos de placas copiadas')

    src_videos = Path(RUTA_REALES_DRIVE) / 'videos'
    if src_videos.exists():
        for vid in src_videos.rglob('*'):
            if vid.suffix.lower() in EXTENSIONES_VIDEO:
                n = extraer_frames(vid, dst_reales)
                print(f'  🎬 {vid.name} → {n} frames')
                total_reales += n

    print(f'\n✅ Total imágenes reales: {total_reales}')
else:
    print('ℹ️  Sin datos reales — entrenando solo con datasets de Kaggle')
    print('   Para agregar videos del condominio:')
    print('   Crea Drive/reales_placa/videos/ y Drive/reales_placa/placas/')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 4 — Organizar dataset en formato YOLO
#
# Estructura final:
#   /content/dataset_placa/
#     ├── images/train/   ← imágenes .jpg
#     ├── images/val/
#     ├── labels/train/   ← anotaciones .txt (YOLO: class cx cy w h)
#     ├── labels/val/
#     └── data.yaml
#
# Clase única: 0 = placa
# Convierte automáticamente Pascal VOC (.xml) a formato YOLO (.txt)
# ─────────────────────────────────────────────────────────────────────────────
import random, shutil, cv2
import xml.etree.ElementTree as ET
from pathlib import Path

random.seed(42)
SPLIT_VAL   = 0.20
PESO_REALES = 4   # repetir imágenes reales N veces (más valiosas que Kaggle)

BASE_DS = Path('/content/dataset_placa')
for split in ['train', 'val']:
    (BASE_DS / 'images' / split).mkdir(parents=True, exist_ok=True)
    (BASE_DS / 'labels' / split).mkdir(parents=True, exist_ok=True)

def xml_a_yolo(xml_path, img_w, img_h):
    """Convierte Pascal VOC XML → formato YOLO (clase cx cy w h normalizados)."""
    boxes = []
    try:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.find('xmin').text)
            ymin = float(bb.find('ymin').text)
            xmax = float(bb.find('xmax').text)
            ymax = float(bb.find('ymax').text)
            cx = ((xmin + xmax) / 2) / img_w
            cy = ((ymin + ymax) / 2) / img_h
            w  = (xmax - xmin) / img_w
            h  = (ymax - ymin) / img_h
            boxes.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    except Exception:
        pass
    return boxes

EXTENSIONES_IMG = {'.jpg', '.jpeg', '.png'}
pares = []

for ds_folder in ['/content/ds_placas1', '/content/ds_placas2', '/content/ds_placas3']:
    folder = Path(ds_folder)
    if not folder.exists():
        continue
    for img in folder.rglob('*'):
        if img.suffix.lower() not in EXTENSIONES_IMG:
            continue
        lbl_yolo = img.with_suffix('.txt')
        lbl_xml  = img.with_suffix('.xml')
        if lbl_yolo.exists():
            pares.append((img, lbl_yolo, 'yolo'))
        elif lbl_xml.exists():
            pares.append((img, lbl_xml, 'xml'))

print(f'📊 Pares imagen+label encontrados: {len(pares)}')
random.shuffle(pares)
split_idx = int(len(pares) * (1 - SPLIT_VAL))

idx = 0
for split_name, items in [('train', pares[:split_idx]), ('val', pares[split_idx:])]:
    for img_path, lbl_path, fmt in items:
        dst_img = BASE_DS / 'images' / split_name / f'placa_{idx:06d}.jpg'
        dst_lbl = BASE_DS / 'labels' / split_name / f'placa_{idx:06d}.txt'
        try:
            img_cv = cv2.imread(str(img_path))
            if img_cv is None:
                continue
            cv2.imwrite(str(dst_img), img_cv)
            if fmt == 'yolo':
                shutil.copy(lbl_path, dst_lbl)
            else:
                h, w = img_cv.shape[:2]
                dst_lbl.write_text('\n'.join(xml_a_yolo(lbl_path, w, h)))
            idx += 1
        except Exception:
            continue

# Agregar imágenes reales con peso extra (sin label = fondo negativo o positivo sin anotar)
n_reales = 0
if TIENE_REALES:
    for img in Path('/content/reales_placa').glob('*.jpg'):
        for _ in range(PESO_REALES):
            dst = BASE_DS / 'images' / 'train' / f'real_{n_reales:06d}.jpg'
            shutil.copy(img, dst)
            (BASE_DS / 'labels' / 'train' / f'real_{n_reales:06d}.txt').write_text('')
            n_reales += 1

(BASE_DS / 'data.yaml').write_text(
    'path: /content/dataset_placa\ntrain: images/train\nval:   images/val\nnc: 1\nnames: [\'placa\']\n'
)

print('\n=== RESUMEN DATASET ===')
for split in ['train', 'val']:
    n = len(list((BASE_DS / 'images' / split).glob('*.jpg')))
    print(f'  {split}: {n} imágenes')
print(f'  Reales añadidas (x{PESO_REALES}): {n_reales}')
print('\n✅ data.yaml creado')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 5 — Fine-tune YOLOv8n para detección de placas
#
# Punto de partida (en orden de preferencia):
#   1. modelo_placa_base.pt  ← tu modelo anterior (Drive)
#   2. keremberke/yolov8n-license-plate-detection  ← HuggingFace Hub
#   3. yolov8n.pt            ← COCO general (fallback)
#
# 30 épocas es suficiente para fine-tuning desde modelo de placas.
# ─────────────────────────────────────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

if Path('/content/modelo_placa_base.pt').exists():
    BASE = '/content/modelo_placa_base.pt'
    print('🔄 Continuando desde modelo_placa.pt (Drive)')
else:
    try:
        from huggingface_hub import hf_hub_download
        BASE = hf_hub_download(
            repo_id='keremberke/yolov8n-license-plate-detection',
            filename='best.pt'
        )
        print('🚀 Partiendo desde modelo pre-entrenado en placas (keremberke/HuggingFace)')
    except Exception as e:
        BASE = 'yolov8n.pt'
        print(f'⚠️  HuggingFace no disponible ({e}), usando yolov8n.pt (COCO — fallback)')

modelo = YOLO(BASE)

print('\n🏋️  Iniciando fine-tuning...')
resultados = modelo.train(
    data    = '/content/dataset_placa/data.yaml',
    epochs  = 30,
    imgsz   = 640,
    batch   = 16,
    patience= 7,           # early stopping: 7 épocas sin mejora
    lr0     = 1e-3,
    lrf     = 1e-2,
    degrees = 10.0,        # rotación: simula cámaras levemente inclinadas
    scale   = 0.5,         # zoom: placa en distintos tamaños
    perspective = 0.001,   # perspectiva: ángulo de cámara variable
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,         # simula noche/día/lluvia
    mosaic  = 1.0,
    mixup   = 0.1,
    project = '/content/runs/placa',
    name    = 'train',
    verbose = False
)

print('\n✅ Fine-tuning completado')
print(f'   Mejor modelo: {resultados.save_dir}/weights/best.pt')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 6 — Evaluar detección de placas (mAP, precisión, recall)
# ─────────────────────────────────────────────────────────────────────────────
from ultralytics import YOLO

MEJOR_PT   = '/content/runs/placa/train/weights/best.pt'
modelo_eval = YOLO(MEJOR_PT)

print('📊 Evaluando sobre conjunto val...')
metricas = modelo_eval.val(
    data   = '/content/dataset_placa/data.yaml',
    imgsz  = 640,
    verbose= False
)

mAP50    = metricas.box.map50
mAP50_95 = metricas.box.map
precision = metricas.box.mp
recall    = metricas.box.mr

print('\n=== MÉTRICAS DE DETECCIÓN YOLO ===')
print(f'  mAP@50:       {mAP50:.3f}  (>0.70 = bueno, >0.85 = excelente)')
print(f'  mAP@50-95:    {mAP50_95:.3f}')
print(f'  Precisión:    {precision:.3f}')
print(f'  Recall:       {recall:.3f}')
print(f'\n  Clase detectada: placa')

if mAP50 >= 0.85:
    print('\n🏆 Excelente — listo para producción')
elif mAP50 >= 0.70:
    print('\n✅ Bueno — funcional, puede mejorar con más imágenes reales del condominio')
else:
    print('\n⚠️  Bajo — agrega más fotos/videos reales del condominio y reentrena')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 7 — Pipeline completo: Detección YOLO + OCR EasyOCR
#
# Formatos bolivianos soportados:
#   1154AER / 4898ELK  → 4 dígitos + 3 letras  (2000 a hoy, más común)
#   824EDH             → 3 dígitos + 3 letras  (1997+)
#   CAL280 / SEK000    → 3 letras  + 3 dígitos (1987-1997)
# ─────────────────────────────────────────────────────────────────────────────
import cv2, re, numpy as np, easyocr
from ultralytics import YOLO

MEJOR_PT = '/content/runs/placa/train/weights/best.pt'
yolo     = YOLO(MEJOR_PT)
reader   = easyocr.Reader(['es', 'en'], gpu=True, verbose=False)
print('✅ YOLO + EasyOCR cargados')

# Formatos bolivianos (sin guion)
PATRON_BO = re.compile(r'^(\d{3,4}[A-Z]{2,3}|[A-Z]{2,3}\d{3,4})$')

def preprocesar_placa(crop):
    """Mejora la imagen de la placa para OCR: contraste adaptativo + denoising."""
    gris     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    mejorado = clahe.apply(gris)
    return cv2.fastNlMeansDenoising(mejorado, h=10)

def limpiar_texto(texto):
    """Normaliza texto OCR: mayúsculas, solo letras y dígitos (Bolivia no usa guion)."""
    return re.sub(r'[^A-Z0-9]', '', texto.upper().strip())

def analizar_placa(frame_bgr, conf_yolo=0.40, conf_ocr=0.50):
    """
    Recibe frame BGR completo. Devuelve lista de placas detectadas:
      [{
        'placa':          '1154AER' | None,
        'texto_raw':      str,
        'confianza_yolo': 0.91,
        'confianza_ocr':  0.85,
        'bbox':           [x1, y1, x2, y2],
        'legible':        True,
        'formato_valido': True
      }]
    """
    resultados_yolo = yolo(frame_bgr, conf=conf_yolo, verbose=False)
    placas = []

    for r in resultados_yolo:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf_y = float(box.conf[0])

            margin = 5
            crop   = frame_bgr[max(0,y1-margin):y2+margin,
                                max(0,x1-margin):x2+margin]
            if crop.size == 0:
                continue

            crop_proc  = preprocesar_placa(crop)
            ocr_result = reader.readtext(
                crop_proc, detail=1,
                allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
            )

            texto_raw = ''.join([r[1] for r in ocr_result])
            texto     = limpiar_texto(texto_raw)
            conf_o    = float(np.mean([r[2] for r in ocr_result])) if ocr_result else 0.0

            es_valida = bool(PATRON_BO.match(texto))
            legible   = conf_o >= conf_ocr and len(texto) >= 6 and es_valida

            placas.append({
                'placa':           texto if legible else None,
                'texto_raw':       texto,
                'confianza_yolo':  round(conf_y, 3),
                'confianza_ocr':   round(conf_o, 3),
                'bbox':            [x1, y1, x2, y2],
                'legible':         legible,
                'formato_valido':  es_valida,
            })

    return placas

print('✅ Pipeline listo. Uso: resultados = analizar_placa(frame_bgr)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 8 — Evaluación visual del pipeline completo
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import random
from pathlib import Path

val_imgs = list(Path('/content/dataset_placa/images/val').glob('*.jpg'))
random.shuffle(val_imgs)
muestra  = val_imgs[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Pipeline Completo: Detección YOLO + OCR EasyOCR', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flatten(), muestra):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    dets = analizar_placa(frame)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    for d in dets:
        x1, y1, x2, y2 = d['bbox']
        color = (0, 200, 0) if d['legible'] else (200, 50, 50)
        cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), color, 2)
        label = d['placa'] or f"({d['texto_raw']}?)"
        cv2.putText(frame_rgb, label, (x1, max(y1-6,10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)

    titulo = ', '.join([d['placa'] or '?' for d in dets]) if dets else 'sin placa'
    ax.imshow(frame_rgb)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Estadísticas sobre 50 imágenes
print('\n=== ESTADÍSTICAS PIPELINE (50 imgs val) ===')
total = legibles = validas = sin_det = 0
for img_path in val_imgs[:50]:
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    dets = analizar_placa(frame)
    total += 1
    if not dets:
        sin_det += 1
    for d in dets:
        if d['legible']:        legibles += 1
        if d['formato_valido']: validas  += 1

print(f'  Imágenes procesadas:   {total}')
print(f'  Sin detección:         {sin_det}')
print(f'  Placas legibles:       {legibles}')
print(f'  Formato BO válido:     {validas}')
if total > 0:
    print(f'  Tasa lectura:          {legibles/total*100:.1f}%')
print('\n  Nota: val contiene placas internacionales (Kaggle), no bolivianas.')
print('  Tasa real con placas BO será significativamente mayor.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 9 — Guardar modelo final + subir a Google Drive
#
# Guarda solo el detector YOLO (modelo_placa.pt).
# EasyOCR NO necesita guardarse — se descarga automáticamente en runtime
# con: easyocr.Reader(['es','en'])
# ─────────────────────────────────────────────────────────────────────────────
import shutil

NOMBRE_FINAL = 'modelo_placa.pt'

shutil.copy('/content/runs/placa/train/weights/best.pt', NOMBRE_FINAL)
print(f'✅ {NOMBRE_FINAL} guardado en /content/')

shutil.copy(NOMBRE_FINAL, f'/content/drive/MyDrive/{NOMBRE_FINAL}')
print(f'✅ Respaldado en Google Drive')

# Descarga directa al computador (descomenta si prefieres):
# from google.colab import files
# files.download(NOMBRE_FINAL)

print(f'\n📋 Resumen del modelo:')
print(f'   Arquitectura:   YOLOv8n fine-tuned para detección de placas')
print(f'   Clase:          0 = placa')
print(f'   OCR:            EasyOCR (sin archivo .pth — se carga en runtime)')
print(f'   mAP@50:         {mAP50:.3f}')
print(f'   Archivo:        {NOMBRE_FINAL}')
print(f'\n⚠️  Coloca modelo_placa.pt en entrenamientopersona/')
print(f'   PlacaDetector lo carga automáticamente desde esa ruta')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELDA 10 — Probar con imagen individual
#
# Sube una foto y llama: probar_imagen('nombre.jpg')
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def probar_imagen(ruta):
    frame = cv2.imread(ruta)
    if frame is None:
        print(f'❌ No se pudo leer: {ruta}')
        return

    dets      = analizar_placa(frame)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(frame_rgb)

    for d in dets:
        x1, y1, x2, y2 = d['bbox']
        color = 'lime' if d['legible'] else 'red'
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=color, facecolor='none'
        ))
        label = f"{d['placa'] or '?'} ({d['confianza_ocr']*100:.0f}%)"
        ax.text(x1, y1-6, label, color=color, fontsize=11, fontweight='bold')

    ax.set_title(f'Placas detectadas: {len(dets)}', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print('\n─── Resultados ───')
    if not dets:
        print('  Sin placas detectadas')
    for i, d in enumerate(dets, 1):
        estado = '✅ LEGIBLE' if d['legible'] else '⚠️  NO LEGIBLE'
        print(f'  [{i}] {estado}')
        print(f'       Placa:           {d["placa"] or "—"}')
        print(f'       Texto raw OCR:   {d["texto_raw"]}')
        print(f'       Confianza YOLO:  {d["confianza_yolo"]*100:.1f}%')
        print(f'       Confianza OCR:   {d["confianza_ocr"]*100:.1f}%')
        print(f'       Formato PE:      {"✅" if d["formato_valido"] else "❌"}')

# Sube una imagen y prueba:
# probar_imagen('mi_placa.jpg')
print('ℹ️  Sube una imagen y llama: probar_imagen("nombre.jpg")')